# GV Tools upper-air sounding download test

This notebook exercises the complete NOAA model workflow: latest download, one requested cycle, an optional cycle range, GRIB2 decoding, and a publication-quality Skew-T/log-P plot. RAP is the operational successor to RUC; NOAA retired the old RUC Soundings service. All times are UTC.


## Install the sounding dependencies

From a `tcsh` terminal, install GV Tools with its optional sounding stack:

```tcsh
set WORKSPACE = "$HOME/Desktop/Work/GV Tools"
set PYTHON = "$HOME/anaconda3/bin/python"
$PYTHON -m pip install "$WORKSPACE/gv_tools/release/gv_tools-0.34.0-py3-none-any.whl[sounding]"
```


In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import gv_tools
from gv_tools.util import (
    download_model_soundings, load_model_profile, plot_skewt,
)

WORKSPACE = Path.home() / 'Desktop' / 'Work' / 'GV Tools'
OUTPUT = WORKSPACE / 'Soundings'
OUTPUT.mkdir(parents=True, exist_ok=True)
LATITUDE, LONGITUDE = 37.94, -75.47  # Wallops Island, Virginia
print('GV Tools:', gv_tools.__version__)
print('Output:', OUTPUT)


GV Tools: 0.34.0
Output: /Users/dwolff/Desktop/Work/GV Tools/Soundings


## Test 1 — latest RAP analysis


In [2]:
latest = download_model_soundings(
    LATITUDE, LONGITUDE, model='rap', latest=True, output_dir=OUTPUT,
)
latest_file = latest[0].path
latest[0]


DownloadedSounding(model='rap', cycle=datetime.datetime(2026, 10, 1, 16, 0, tzinfo=datetime.timezone.utc), forecast_hour=0, valid_time=datetime.datetime(2026, 10, 1, 16, 0, tzinfo=datetime.timezone.utc), latitude=37.94, longitude=-75.47, path=PosixPath('/Users/dwolff/Desktop/Work/GV Tools/Soundings/rap_20261001_16z_f000_+037.940_-0075.470.grib2'), url='https://nomads.ncep.noaa.gov/cgi-bin/filter_rap.pl?file=rap.t16z.awp130pgrbf00.grib2&all_lev=on&var_HGT=on&var_RH=on&var_TMP=on&var_UGRD=on&var_VGRD=on&subregion=&toplat=38.0900&leftlon=-75.6200&rightlon=-75.3200&bottomlat=37.7900&dir=%2Frap.20261001')

## Decode and plot the latest profile


In [3]:
profile = load_model_profile(
    latest_file, latitude=LATITUDE, longitude=LONGITUDE,
)
plot_path = OUTPUT / f'{latest_file.stem}_skewt.png'
figure = plot_skewt(
    profile, plot_path, title='Wallops Island RAP analysis', dpi=220,
)
print(plot_path)
print(figure.gv_tools_sounding_diagnostics)
figure


ValueError: unrecognized engine 'cfgrib' must be one of your download engines: ['netcdf4', 'h5netcdf', 'scipy', 'cfradial1', 'datamet', 'furuno', 'gamic', 'gini', 'hpl', 'iris', 'metek', 'nexradlevel2', 'odim', 'radrs-raystack', 'radrs-xradar', 'rainbow', 'store', 'uf']. To install additional dependencies, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

## Test 2 — one explicit cycle

Use the cycle selected by the latest test so this cell remains valid while that cycle is retained by NOMADS. Existing files are reused by default.


In [ ]:
single = download_model_soundings(
    LATITUDE, LONGITUDE, model='rap', when=latest[0].cycle, output_dir=OUTPUT,
)
assert single[0].path == latest_file
single[0]


## Test 3 — short range

This requests the latest three hourly RAP cycles. Set `RUN_RANGE_TEST` to `False` when only one network request is desired.


In [ ]:
RUN_RANGE_TEST = True
if RUN_RANGE_TEST:
    range_end = latest[0].cycle
    range_start = range_end - timedelta(hours=2)
    ranged = download_model_soundings(
        LATITUDE, LONGITUDE, model='rap',
        start=range_start, end=range_end, output_dir=OUTPUT,
    )
    print(f'Downloaded or reused {len(ranged)} cycles')
    for item in ranged:
        print(item.cycle, item.path.name)


## Command-line equivalents

```tcsh
$HOME/anaconda3/bin/gv-tools-sounding 37.94 -75.47 --model rap --latest --output-dir Soundings
$HOME/anaconda3/bin/python examples/plot_model_skewt.py Soundings/profile.grib2 Soundings/skewt.png --latitude 37.94 --longitude -75.47
```
